In [ ]:
from dataclasses import replace, asdict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathtracer import Renderer, lattice_preview, display_encode

from stats import pixel_distribution, principal_components
from spectral import lattice_geometry, row_twiddle, fft, bin_freq, alias_radius_sq

---
# DATA
Renders the Cornell box once and tabulates each AOV's shape, dtype and range next to the camera that produced it.

- `beauty` and `luminance` are scene-referred linear values in cd/m^2, not display codes
- `rendercam` is the single source of camera state: later sections read their lattice geometry or exposure from it
- Change the camera or AOVs here and rerun all; no later cell renders again

In [ ]:
scene = Renderer("scenes/cornell.json")
rendercam = replace(scene.default_camera)
frame = scene.render(camera=rendercam, aovs=("beauty", "luminance"))

In [ ]:
stats = {k: dict(shape=v.shape, dtype=v.dtype, min=v.min(), max=v.max(), mean=v.mean()) for k, v in frame.items()}
display(pd.DataFrame(stats).T, pd.Series(asdict(rendercam), name="camera").to_frame())

---
# IMAGE
Beauty as the viewer displays it: lattice preview, 8-bit code statistics and each RGB channel in grayscale.

- `lattice_preview` draws each pixel as its true cell (hexagon or square), so the hex row offset is visible
- Stats and channels use `display_encode` output: camera exposure, OCIO display transform, dither, then 8-bit quantise
- Statistics therefore describe the displayed image, not scene radiance; dither fills codes a plain quantise would leave empty

In [ ]:
x = frame["beauty"]

plt.imshow(display_encode(lattice_preview(x, rendercam, scale=8), camera=rendercam))
plt.xticks([]), plt.yticks([])
plt.tight_layout()

In [ ]:
# 8-bit stats of beauty as the viewer displays it: pixel_distribution bins codes 0-255, so it takes display_encode output, not cd/m^2
encoded = display_encode(x, camera=rendercam)

pixel_count, stat = pixel_distribution(encoded)

labels = ("shape", "pixels", "min", "max", "least_frequent", "most_frequent", "mean", "std", "median", "nonzero", "empty_bins", "unique_colours")
display(pd.Series((encoded.shape, *stat), index=labels, name="beauty").to_frame())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=[18, 5])
for c, name in enumerate(("R", "G", "B")):
    ax[c].imshow(encoded[..., c], cmap="gray")
    ax[c].set_title(name)
    ax[c].set_xticks([]), ax[c].set_yticks([])
plt.tight_layout()

---
# SPECTRAL
Luminance spectrum on the camera's pixel lattice: where energy sits by frequency and direction, and how far above the noise floor.

- Lattice-aware DFT: a per-row twiddle moves odd rows to their true positions, so square and hex share one path
- Frequencies are cycles per column pitch; each bin is folded to its nearest reciprocal-lattice point for its true radius and angle
- Radial power excludes DC and stops at the lattice Nyquist; the spatial frequency curve is f·P(f), so its peak is the dominant scale
- Phase is referenced to the luminance centroid, so it does not depend on where the scene sits in frame
- Log magnitude is dB re the peak bin; zero bins are NaN and left out of the noise floor histogram

In [ ]:
row_pitch, row_shift = lattice_geometry(rendercam)
img = frame["luminance"][..., 0]
twiddle = row_twiddle(img.shape, row_shift)
spectrum = fft(img, twiddle)
fy, fx = bin_freq(img.shape, row_pitch)
radius_sq, true_fy, true_fx = alias_radius_sq(fy, fx, row_pitch, row_shift)
radius = np.sqrt(radius_sq)
mag = np.abs(spectrum)

bin_width = 1.0 / min(img.shape[1], img.shape[0] * row_pitch)
nyquist = 0.5 * min(1.0 / row_pitch, np.hypot(1.0, row_shift / row_pitch))
# DC excluded
power, edges = np.histogram(radius, bins=np.arange(0.0, radius.max() + bin_width, bin_width), weights=np.where(radius > 0, mag**2, 0.0))
centres = 0.5 * (edges[1:] + edges[:-1])
inside = edges[1:] <= nyquist
log_power = centres * power
peak = np.argmax(np.where(inside, log_power, 0.0))
print(f"Frequency at which power per log frequency is greatest: {centres[peak]:.4f} cycles/pitch")

total = img.sum()
cx = (np.arange(img.shape[1]) @ img.sum(axis=0) + row_shift * img[1::2].sum()) / total
cy = row_pitch * (np.arange(img.shape[0]) @ img.sum(axis=1)) / total
phase = np.angle(spectrum * np.exp(2j * np.pi * (fx[None, :] * cx + fy[:, None] * cy)))
log_mag = 20 * np.log10(mag / mag.max(), out=np.full_like(mag, np.nan), where=mag > 0)

In [ ]:
fig, (ax_mag, ax_phase) = plt.subplots(1, 2, figsize=[20,10])
ax_mag.imshow(np.fft.fftshift(log_mag), cmap="gray")
ax_mag.set_title("Log magnitude")

ax_phase.imshow(np.fft.fftshift(phase), cmap="gray")
ax_phase.set_title("Phase")

In [ ]:
power_2d = mag**2
theta = np.arctan2(true_fy, true_fx)

fig, ax = plt.subplots(1, 5, figsize=[25, 5])

ax[0].plot(centres[inside], log_power[inside], lw=1)
ax[0].axvline(centres[peak], color="r", lw=1)
ax[0].set_title("Spatial frequency")
ax[0].set_xlabel("Radius (cycles/pitch)"), ax[0].set_ylabel("Power per log frequency")
ax[0].set_ylim(bottom=0)

ax[1].plot(edges[1:], np.cumsum(power) / power.sum(), lw=1)
ax[1].set_title("Cumulative energy")
ax[1].set_xlabel("Radius (cycles/pitch)"), ax[1].set_ylabel("Fraction of AC power")

ax[2].semilogy(np.fft.fftshift(fx), np.fft.fftshift(power_2d.sum(axis=0)), lw=1, label="fx")
ax[2].semilogy(np.fft.fftshift(fy), np.fft.fftshift(power_2d.sum(axis=1)), lw=1, label="fy")
ax[2].set_title("Marginal spectra")
ax[2].set_xlabel("Frequency (cycles/pitch)"), ax[2].set_ylabel("Power")
ax[2].legend()

ax[3].hist(log_mag[np.isfinite(log_mag)], bins=100)
ax[3].set_title("Noise floor")
ax[3].set_xlabel("Magnitude (dB re peak)"), ax[3].set_ylabel("Bins")

ring = (radius > 0) & (radius <= nyquist)
ax[4].hist(theta[ring], bins=36, range=(-np.pi, np.pi), weights=power_2d[ring])
ax[4].set_title("Angular power")
ax[4].set_xlabel("Angle (rad)"), ax[4].set_ylabel("Power")

for a in ax: a.grid(alpha=0.25)

---
# PCA
Principal axes of beauty's RGB values: how much colour variance each axis carries and where in the image it lives.

- Runs on linear beauty, not the encoded image, so axes reflect radiance rather than display gamma
- The nearer PC1 is to (1, 1, 1)/√3, the more variation is brightness; PC2 and PC3 then carry chroma
- Projection is each pixel's coordinate along that axis; distance is its distance from that axis (norm over the other two)

In [ ]:
_, variance, axes, projection = principal_components(frame["beauty"])
display(pd.DataFrame(axes.T, index=("PC1", "PC2", "PC3"), columns=("R", "G", "B")).assign(variance=variance, explained=variance / variance.sum()))

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=[18, 7])
for i in range(3):
    ax[0, i].imshow(projection[..., i])
    ax[0, i].set_title(f"PC{i + 1} projection")
    distance = np.linalg.norm(np.delete(projection, i, axis=-1), axis=-1)
    ax[1, i].imshow(distance, cmap="gray")
    ax[1, i].set_title(f"PC{i + 1} distance")
for a in ax.flat: a.set_xticks([]), a.set_yticks([])
plt.tight_layout()